# UMAMI Urban Heat V2 - All-in-One Colab Training

11-channel ResNet34 U-Net for T_mrt prediction.
Channels: 5 base (DSM, CDSM, Albedo, solar x2) + SVF + density + distance + aspect(2) + shadow = 11

Server CPU was stuck at epoch 12/200 after 10+ days. Colab A100 completes in ~3-4 hours.

In [ ]:
import torch, torch.nn as nn, torchvision.models as models
import numpy as np, os, json, time, math, re, tarfile, shutil, gc
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from pathlib import Path
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
!pip install onnx onnxruntime onnxscript rasterio -q
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/UMAMI/urban_heat_v2'
LOCAL = '/content/uh_data'
os.makedirs(LOCAL, exist_ok=True)
for f in sorted(os.listdir(DRIVE_DIR)):
    print(f'  {f}: {os.path.getsize(os.path.join(DRIVE_DIR, f))/1e6:.0f} MB')

PyTorch: 2.10.0+cu128, CUDA: True
GPU: NVIDIA A100-SXM4-80GB, VRAM: 85.1 GB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 127.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 134.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 19.0 MB/s eta 0:00:00
Mounted at /content/drive
  UMAMI_UrbanHeat_V2_Training.ipynb: 0 MB
  dataset_stats.json: 0 MB
  uh_v2_checkpoint_ep12.pt: 293 MB
  uh_v2_features.tar.gz: 8433 MB
  uh_v2_patches (1).tar.gz: 6450 MB
  uh_v2_patches.tar.gz: 6450 MB


In [ ]:
# Step 2: Unpack data to local SSD
PATCHES_DIR = os.path.join(LOCAL, 'training', 'patches')
if not os.path.exists(PATCHES_DIR):
    for arc in ['uh_v2_patches.tar.gz', 'uh_v2_features.tar.gz']:
        t0 = time.time()
        print(f'Unpacking {arc}...', end=' ')
        with tarfile.open(os.path.join(DRIVE_DIR, arc), 'r:gz') as tar:
            tar.extractall(LOCAL, filter='data')
        print(f'{time.time()-t0:.0f}s')
else:
    print('Already unpacked')
stats_dst = os.path.join(LOCAL, 'dataset_stats.json')
if not os.path.exists(stats_dst):
    shutil.copy2(os.path.join(DRIVE_DIR, 'dataset_stats.json'), stats_dst)
with open(stats_dst) as f:
    stats = json.load(f)
splits = stats['splits']
patch_dirs = [d for d in os.listdir(PATCHES_DIR) if os.path.isdir(os.path.join(PATCHES_DIR, d))]
npy_files = [f for f in os.listdir(PATCHES_DIR) if f.endswith('.npy')]
print(f'Patches: {len(patch_dirs)} dirs, {len(npy_files)} .npy files')
print(f'Train: {splits["train_count"]}, Val: {splits["val_count"]}, Test: {splits["test_count"]}')

Unpacking uh_v2_patches.tar.gz... 106s
Unpacking uh_v2_features.tar.gz... 172s
Patches: 1356 dirs, 17275 .npy files
Train: 789, Val: 98, Test: 100


In [ ]:
# Step 3: Dataset with solar position
def compute_solar(lat, lon, doy, hour, utc_offset=10):
    sh = hour - utc_offset + lon / 15.0
    dec = -23.45 * math.cos(math.radians(360/365*(doy+10)))
    ha = 15 * (sh - 12)
    lr, dr, hr = math.radians(lat), math.radians(dec), math.radians(ha)
    sa = math.sin(lr)*math.sin(dr) + math.cos(lr)*math.cos(dr)*math.cos(hr)
    alt = math.degrees(math.asin(max(-1, min(1, sa))))
    if math.cos(math.radians(alt)) < 0.001: return alt, 180.0
    ca = (math.sin(dr) - math.sin(lr)*sa) / (math.cos(lr)*math.cos(math.radians(alt)))
    az = math.degrees(math.acos(max(-1, min(1, ca))))
    if ha > 0: az = 360 - az
    return alt, az

SOLAR = {}
for d,h in [(356,9),(356,11),(356,13),(356,15),(15,9),(15,11),(15,13),(15,15),(46,9),(46,13),(306,9),(306,13)]:
    SOLAR[(d,h)] = compute_solar(-37.81, 144.96, d, h)
TRE = re.compile(r'_d(\d{3})_h(\d{2})_target\.npy$')
EXTRA = ['svf','building_density','tree_distance','aspect','shadow_fraction']
IN_CH = 11  # 5 base + 6 extra (aspect=2ch)

class UHDS(Dataset):
    def __init__(self, pdir, names, feats, aug=False):
        self.pdir, self.feats, self.aug = Path(pdir), feats, aug
        self.pairs = []
        for bn in names:
            for v in [bn, f'{bn}_green', f'{bn}_cool']:
                ip = self.pdir / f'{v}_input.npy'
                if not ip.exists(): continue
                eps = []
                ok = True
                for ft in self.feats:
                    fp = self.pdir / f'{v}_{ft}.npy'
                    if not fp.exists(): ok = False; break
                    eps.append(fp)
                if not ok and self.feats: continue
                for tf in sorted(self.pdir.glob(f'{v}_d*_target.npy')):
                    m = TRE.search(tf.name)
                    if not m: continue
                    doy, hr = int(m.group(1)), int(m.group(2))
                    alt, az = SOLAR.get((doy,hr), compute_solar(-37.81, 144.96, doy, hr))
                    self.pairs.append((ip, tf, alt, az, eps))
        print(f'  Dataset: {len(self.pairs)} pairs from {len(names)} patches')
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        ip, tp, alt, az, eps = self.pairs[i]
        x = torch.from_numpy(np.load(ip).astype(np.float32))
        y = torch.from_numpy(np.load(tp).astype(np.float32))
        h, w = x.shape[1], x.shape[2]
        x = torch.cat([x,
            torch.full((1,h,w), math.sin(math.radians(max(alt,0))), dtype=torch.float32),
            torch.full((1,h,w), max(alt,0)/90.0, dtype=torch.float32)], dim=0)
        for fp in eps:
            x = torch.cat([x, torch.from_numpy(np.load(fp).astype(np.float32))], dim=0)
        if self.aug:
            if torch.rand(1).item() > 0.5: x, y = torch.flip(x,[2]), torch.flip(y,[2])
            if torch.rand(1).item() > 0.5: x, y = torch.flip(x,[1]), torch.flip(y,[1])
        return x, y

print('Loading datasets...')
tds = UHDS(PATCHES_DIR, splits['train_patches'], EXTRA, aug=True)
vds = UHDS(PATCHES_DIR, splits['val_patches'], EXTRA)
x0,y0 = tds[0]
print(f'Input: {x0.shape}, Target: {y0.shape}')
assert x0.shape[0] == IN_CH, f'Expected {IN_CH}, got {x0.shape[0]}'

Loading datasets...
  Dataset: 8004 pairs from 789 patches
  Dataset: 396 pairs from 98 patches
Input: torch.Size([11, 512, 512]), Target: torch.Size([1, 512, 512])


In [ ]:
# Step 4: Model
class ResEnc(nn.Module):
    def __init__(self, ic=5, pt=True):
        super().__init__()
        r = models.resnet34(weights=models.ResNet34_Weights.DEFAULT if pt else None)
        self.c1 = nn.Conv2d(ic, 64, 7, stride=2, padding=3, bias=False)
        if pt:
            with torch.no_grad(): self.c1.weight[:,:3] = r.conv1.weight.data; self.c1.weight[:,3:] = 0
        self.bn1 = nn.GroupNorm(8, 64)
        self.relu = nn.ReLU(True)
        self.mp = nn.MaxPool2d(3, stride=2, padding=1)
        def rbn(ly):
            for n, m in ly.named_modules():
                if isinstance(m, nn.BatchNorm2d):
                    p = ly
                    for s in n.split('.')[:-1]: p = getattr(p, s)
                    setattr(p, n.split('.')[-1], nn.GroupNorm(8, m.num_features))
            return ly
        self.l1,self.l2,self.l3,self.l4 = rbn(r.layer1),rbn(r.layer2),rbn(r.layer3),rbn(r.layer4)
    def forward(self, x):
        e0 = self.relu(self.bn1(self.c1(x)))
        return [e0, self.l1(self.mp(e0)), self.l2(self.l1(self.mp(e0))),
                self.l3(self.l2(self.l1(self.mp(e0)))),
                self.l4(self.l3(self.l2(self.l1(self.mp(e0)))))]

# Use the cleaner version
class ResNetEncoder(nn.Module):
    def __init__(self, ic=5, pt=True):
        super().__init__()
        r = models.resnet34(weights=models.ResNet34_Weights.DEFAULT if pt else None)
        self.c1 = nn.Conv2d(ic, 64, 7, stride=2, padding=3, bias=False)
        if pt:
            with torch.no_grad(): self.c1.weight[:,:3] = r.conv1.weight.data; self.c1.weight[:,3:] = 0
        self.bn1 = nn.GroupNorm(8, 64)
        self.relu = nn.ReLU(True)
        self.mp = nn.MaxPool2d(3, stride=2, padding=1)
        def rbn(ly):
            for n, m in ly.named_modules():
                if isinstance(m, nn.BatchNorm2d):
                    parts = n.split('.')
                    p = ly
                    for s in parts[:-1]: p = getattr(p, s)
                    setattr(p, parts[-1], nn.GroupNorm(8, m.num_features))
            return ly
        self.l1 = rbn(r.layer1); self.l2 = rbn(r.layer2)
        self.l3 = rbn(r.layer3); self.l4 = rbn(r.layer4)
    def forward(self, x):
        e0 = self.relu(self.bn1(self.c1(x)))
        e1 = self.l1(self.mp(e0)); e2 = self.l2(e1)
        e3 = self.l3(e2); e4 = self.l4(e3)
        return [e0, e1, e2, e3, e4]

class DecBlock(nn.Module):
    def __init__(self, ic, sc, oc):
        super().__init__()
        self.up = nn.ConvTranspose2d(ic, ic//2, 2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(ic//2+sc, oc, 3, padding=1, bias=False), nn.GroupNorm(8, oc), nn.ReLU(True),
            nn.Conv2d(oc, oc, 3, padding=1, bias=False), nn.GroupNorm(8, oc), nn.ReLU(True))
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = nn.functional.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        return self.conv(torch.cat([x, skip], 1))

class UNetR34(nn.Module):
    def __init__(self, ic=5, oc=1, pt=True):
        super().__init__()
        self.enc = ResNetEncoder(ic, pt)
        self.d4 = DecBlock(512,256,256); self.d3 = DecBlock(256,128,128)
        self.d2 = DecBlock(128,64,64); self.d1 = DecBlock(64,64,64)
        self.fu = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.fc = nn.Sequential(nn.Conv2d(32,32,3,padding=1,bias=False),nn.GroupNorm(8,32),nn.ReLU(True),
                                nn.Conv2d(32,oc,1),nn.Sigmoid())
    def forward(self, x):
        e0,e1,e2,e3,e4 = self.enc(x)
        return self.fc(self.fu(self.d1(self.d2(self.d3(self.d4(e4,e3),e2),e1),e0)))

class PercLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        self.s1 = nn.Sequential(*list(vgg.features[:4]))
        self.s2 = nn.Sequential(*list(vgg.features[4:9]))
        self.s3 = nn.Sequential(*list(vgg.features[9:16]))
        for p in self.parameters(): p.requires_grad = False
        self.w = [1.0, 0.5, 0.25]
    def forward(self, p, t):
        p3, t3 = p.expand(-1,3,-1,-1), t.expand(-1,3,-1,-1)
        loss = 0.0
        for i, ly in enumerate([self.s1, self.s2, self.s3]):
            p3, t3 = ly(p3), ly(t3)
            loss += self.w[i] * nn.functional.l1_loss(p3, t3)
        return loss

model = UNetR34(IN_CH, 1, True).to(device)
perc = PercLoss().to(device); perc.eval()
params = sum(p.numel() for p in model.parameters())
print(f'UNetResNet34 V2: {params:,} params')

# Warm-start option
USE_CHECKPOINT = False  # Set True to continue from epoch 12
ckpt = os.path.join(DRIVE_DIR, 'uh_v2_checkpoint_ep12.pt')
if USE_CHECKPOINT and os.path.exists(ckpt):
    ck = torch.load(ckpt, map_location='cpu', weights_only=False)
    model.load_state_dict(ck['model']); print(f'Loaded epoch {ck.get("epoch","?")} checkpoint')
    del ck
else:
    print('Fresh training (ImageNet backbone)')

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 195MB/s]


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 243MB/s]


UNetResNet34 V2: 24,440,737 params
Fresh training (ImageNet backbone)


In [ ]:
# Step 5: Training
BS = 16 if torch.cuda.get_device_properties(0).total_memory > 40e9 else 8
tdl = DataLoader(tds, BS, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
vdl = DataLoader(vds, BS, num_workers=2, pin_memory=True)
print(f'Train: {len(tds):,} ({len(tdl)} batches), Val: {len(vds):,}, BS={BS}')

EP, LR, PW = 200, 2e-4, 0.1
TMRT = stats['normalization']['tmrt_range'][1]
opt = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.5, 0.999))
sched = CosineAnnealingWarmRestarts(opt, T_0=20, T_mult=2, eta_min=1e-6)
l1 = nn.L1Loss()
CKDIR = os.path.join(DRIVE_DIR, 'checkpoints_v2'); os.makedirs(CKDIR, exist_ok=True)

se, br, hist = 0, float('inf'), []
rp = os.path.join(CKDIR, 'latest_model.pt')
if os.path.exists(rp):
    ck = torch.load(rp, map_location=device, weights_only=False)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['optimizer'])
    sched.load_state_dict(ck['scheduler']); se = ck['epoch']+1
    br = ck.get('best_val_rmse', float('inf')); hist = ck.get('history', [])
    print(f'Resume epoch {se}, best RMSE: {br:.3f}')
    del ck
else:
    print(f'Fresh: {EP} epochs, LR={LR}')

Train: 8,004 (500 batches), Val: 396, BS=16
Fresh: 200 epochs, LR=0.0002


In [ ]:
def val_metrics(model, dl, tmrt):
    model.eval(); ps, ts = [], []
    with torch.no_grad():
        for xb, yb in dl:
            ps.append((model(xb.to(device)).cpu()*tmrt)); ts.append(yb*tmrt)
    p, t = torch.cat(ps).numpy().flatten(), torch.cat(ts).numpy().flatten()
    rmse = float(np.sqrt(np.mean((p-t)**2)))
    mae = float(np.mean(np.abs(p-t)))
    ss_r, ss_t = np.sum((t-p)**2), np.sum((t-np.mean(t))**2)
    r2 = float(1-ss_r/max(ss_t,1e-8))
    del ps, ts; gc.collect(); torch.cuda.empty_cache()
    return {'rmse': rmse, 'mae': mae, 'r2': r2}

print(f"{'Ep':>4} {'TrL':>8} {'RMSE':>7} {'MAE':>7} {'R2':>6} {'LR':>9} {'T':>5}")
print('-'*60)
for epoch in range(se, EP):
    t0 = time.time()
    model.train(); tl = 0
    for xb, yb in tdl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        p = model(xb)
        loss = l1(p, yb) + PW * perc(p, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); tl += loss.item()
    tl /= len(tdl); sched.step()
    # Validate every 5 epochs
    m = val_metrics(model, vdl, TMRT) if epoch % 5 == 0 or epoch == EP-1 else None
    lr = opt.param_groups[0]['lr']; imp = ''
    if m and m['rmse'] < br:
        br = m['rmse']
        torch.save({'epoch':epoch,'model':model.state_dict(),'optimizer':opt.state_dict(),
            'scheduler':sched.state_dict(),'best_val_rmse':br,'metrics':m,
            'in_channels':IN_CH,'extra_features':EXTRA}, os.path.join(CKDIR,'best_model.pt'))
        imp = ' *'
    if epoch % 10 == 0 or epoch == EP-1:
        torch.save({'epoch':epoch,'model':model.state_dict(),'optimizer':opt.state_dict(),
            'scheduler':sched.state_dict(),'best_val_rmse':br,'history':hist},
            os.path.join(CKDIR,'latest_model.pt'))
    if m:
        print(f"{epoch:4d} {tl:8.5f} {m['rmse']:7.3f} {m['mae']:7.3f} {m['r2']:6.3f} {lr:9.2e} {time.time()-t0:.0f}s{imp}")
    else:
        print(f"{epoch:4d} {tl:8.5f} {'':>7} {'':>7} {'':>6} {lr:9.2e} {time.time()-t0:.0f}s")
    hist.append({'epoch':epoch,'tl':tl,'lr':lr,**(m or {})})
print(f'\nBest RMSE: {br:.3f} C')

  Ep      TrL    RMSE     MAE     R2        LR     T
------------------------------------------------------------
   0  0.11106   6.684   4.531  0.236  1.99e-04 137s *
   1  0.10256                         1.95e-04 126s
   2  0.09979                         1.89e-04 126s
   3  0.09723                         1.81e-04 126s
   4  0.09458                         1.71e-04 126s
   5  0.09287   6.243   4.274  0.334  1.59e-04 135s *
   6  0.09157                         1.46e-04 126s
   7  0.09234                         1.31e-04 127s
   8  0.08742                         1.16e-04 127s
   9  0.08355                         1.01e-04 126s
  10  0.08138   5.671   3.229  0.450  8.49e-05 136s *
  11  0.07872                         6.98e-05 126s
  12  0.07520                         5.53e-05 126s
  13  0.07178                         4.20e-05 126s
  14  0.06848                         3.01e-05 127s
  15  0.06585   5.342   2.489  0.512  2.00e-05 135s *
  16  0.06407                         1.18e-05

In [ ]:
# Step 6: ONNX Export
import onnx, onnxruntime as ort
ck = torch.load(os.path.join(CKDIR,'best_model.pt'), map_location='cpu', weights_only=False)
model.cpu().load_state_dict(ck['model']); model.eval()
print(f'Best epoch {ck["epoch"]}: RMSE={ck["metrics"]["rmse"]:.3f}, MAE={ck["metrics"]["mae"]:.3f}, R2={ck["metrics"]["r2"]:.3f}')

op = os.path.join(DRIVE_DIR, 'umami_unet_v2_fp32.onnx')
torch.onnx.export(model, torch.randn(1,IN_CH,512,512), op, opset_version=17,
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input':{0:'batch'},'output':{0:'batch'}}, do_constant_folding=True)
print(f'Saved: {op} ({os.path.getsize(op)/1e6:.1f} MB)')

s = ort.InferenceSession(op, providers=['CPUExecutionProvider'])
xt = np.random.randn(1,IN_CH,512,512).astype(np.float32)
oo = s.run(None, {'input': xt})[0]
with torch.no_grad(): po = model(torch.from_numpy(xt)).numpy()
print(f'Verify: max diff={np.abs(po-oo).max():.8f}')
print(f'Range: [{oo.min():.4f}, {oo.max():.4f}]')

rpt = {'model':'V2 (11ch)','in_channels':IN_CH,'extra_features':EXTRA,'params':params,
    'best_epoch':ck['epoch'],'best_metrics':ck['metrics'],'total_epochs':len(hist),
    'batch_size':BS,'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'history':hist}
with open(os.path.join(DRIVE_DIR,'v2_report.json'),'w') as f: json.dump(rpt,f,indent=2)
print('Report saved. Deploy: rclone copy umami_unet_v2_fp32.onnx to server.')

Best epoch 135: RMSE=4.766, MAE=1.675, R2=0.612


/tmp/ipykernel_30048/891435496.py:8: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(model, torch.randn(1,IN_CH,512,512), op, opset_version=17,
W0328 14:47:11.786000 30048 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0328 14:47:12.581000 30048 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', 

[torch.onnx] Obtain model graph for `UNetR34([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `UNetR34([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/BaseConverter.h:64: adapter_lookup: Assertion `false`

Applied 45 of general pattern rewrite rules.
Saved: /content/drive/MyDrive/UMAMI/urban_heat_v2/umami_unet_v2_fp32.onnx (0.6 MB)
Verify: max diff=0.00001699
Range: [0.1604, 0.5106]
Report saved. Deploy: rclone copy umami_unet_v2_fp32.onnx to server.


In [ ]:
# Backup patches to Drive (safety net)
# Only needed if you want to re-run without re-extracting
# Uncomment to run:
# !cp -r /content/uh_data/training /content/drive/MyDrive/UMAMI/urban_heat_v2/training_backup/